In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

!pip install torchmetrics
from torchmetrics import Accuracy

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [11]:
split = torch.load("indices.pt")
train_indices = split["train_indices"]
test_indices  = split["test_indices"]

data = torch.load("allData_10s.pt", map_location=device)

waveforms = data["raw waves"]
labels =    data["depth labels"]

train_waveforms = waveforms[train_indices]
train_labels    = labels[train_indices]

test_waveforms = waveforms[test_indices]
test_labels    = labels[test_indices]

train_dataset = TensorDataset(train_waveforms, train_labels)
test_dataset  = TensorDataset(test_waveforms, test_labels)

training = DataLoader(train_dataset, batch_size=16, shuffle=True)
testing  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

In [12]:
classes = 4

# Define Model
class depthCNN(nn.Module):

    def __init__(self, num_features):
        super(depthCNN, self).__init__()

        self.conv1   = nn.Conv2d(1,   out_channels = 64,  kernel_size = (3,7), padding =1)
        self.conv2   = nn.Conv2d(64,  out_channels = 64,  kernel_size = (3,7), padding =1)
        self.conv3   = nn.Conv2d(64,  out_channels = 64,  kernel_size = (3,7), padding =1)
        self.conv4   = nn.Conv2d(64,  out_channels = 64,  kernel_size = (3,7), padding =1)
        self.conv5   = nn.Conv2d(64,  out_channels = 64,  kernel_size = (3,7), padding =1)
        # self.conv6   = nn.Conv2d(64,  out_channels = 64,  kernel_size = (3,7), padding =1)
        self.pool    = nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 1)
        self.bn1     = nn.BatchNorm2d(64)
        self.bn2     = nn.BatchNorm2d(64)
        self.bn3     = nn.BatchNorm2d(64)
        self.bn4     = nn.BatchNorm2d(64)
        self.bn5     = nn.BatchNorm2d(64)
        self.bn6     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(p=0.5)

        self.fullyconnected = nn.LazyLinear(classes)

    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = self.dropout(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = self.dropout(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = self.pool(x)
        # x = self.dropout(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.relu(x)
        x = self.pool(x)

        # x = self.conv6(x)
        # x = self.bn6(x)
        # x = self.relu(x)
        # x = self.pool(x)

        x = self.dropout(x)

        x = torch.flatten(x, start_dim = 1)
        x = self.fullyconnected(x)
        x = self.softmax(x)

        return x

In [13]:
# Construct model
model = depthCNN(classes)
model.to(device)

# define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001) # 0.001
criterion = nn.CrossEntropyLoss()

In [14]:
epochs = 250
# Training
for epoch in range(epochs):

    print(f"Epoch [{epoch + 1}/{epochs}]")

    for i, data in enumerate(training):

        raw_data, labels = data

        labels = torch.squeeze(labels)

        depth_class = model(raw_data)

        loss = criterion(depth_class, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Epoch [1/250]
Epoch [2/250]
Epoch [3/250]
Epoch [4/250]
Epoch [5/250]
Epoch [6/250]
Epoch [7/250]
Epoch [8/250]
Epoch [9/250]
Epoch [10/250]
Epoch [11/250]
Epoch [12/250]
Epoch [13/250]
Epoch [14/250]
Epoch [15/250]
Epoch [16/250]
Epoch [17/250]
Epoch [18/250]
Epoch [19/250]
Epoch [20/250]
Epoch [21/250]
Epoch [22/250]
Epoch [23/250]
Epoch [24/250]
Epoch [25/250]
Epoch [26/250]
Epoch [27/250]
Epoch [28/250]
Epoch [29/250]
Epoch [30/250]
Epoch [31/250]
Epoch [32/250]
Epoch [33/250]
Epoch [34/250]
Epoch [35/250]
Epoch [36/250]
Epoch [37/250]
Epoch [38/250]
Epoch [39/250]
Epoch [40/250]
Epoch [41/250]
Epoch [42/250]
Epoch [43/250]
Epoch [44/250]
Epoch [45/250]
Epoch [46/250]
Epoch [47/250]
Epoch [48/250]
Epoch [49/250]
Epoch [50/250]
Epoch [51/250]
Epoch [52/250]
Epoch [53/250]
Epoch [54/250]
Epoch [55/250]
Epoch [56/250]
Epoch [57/250]
Epoch [58/250]
Epoch [59/250]
Epoch [60/250]
Epoch [61/250]
Epoch [62/250]
Epoch [63/250]
Epoch [64/250]
Epoch [65/250]
Epoch [66/250]
Epoch [67/250]
Epoc

In [15]:
# Testing
acc = Accuracy(task="multiclass", num_classes = classes).to(device)
model.eval()
with torch.no_grad():
   for i, data in enumerate(testing):

       raw_data, labels = data
       labels = torch.squeeze(labels)
       depth_class = model(raw_data)
       _, preds = torch.max(depth_class, 1)
       _, truth = torch.max(labels, 1)

       acc(preds, truth)

test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")


Test accuracy: 0.785345733165741


In [16]:
torch.save(model.state_dict(), "depth_allData_10s.pth")